In [0]:
#Import libraries
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType, IntegerType
from pyspark.sql.functions import trim,col

In [0]:
# Read data bronze table and load to dataframe
df = spark.table("baraa_projectwork.bronze.erp_px_cat_g1v2")
display(df)


In [0]:
# Remove extra spaces from all string columns
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
    df = df.withColumn(field.name, trim(col(field.name)))

In [0]:

df = df.withColumn(
    "maintenance",
    F.when(F.upper(col("maintenance")) == "YES", F.lit(True))
     .when(F.upper(col("maintenance")) == "NO", F.lit(False))
     .otherwise(None)
)

In [0]:
RENAME_MAP = {
    "ID": "category_id",
    "CAT": "category",
    "SUBCAT": "sub_category",
    "maintenance": "maintenance_flag"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
df.limit(10).display()

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("baraa_projectwork.silver.erp_product_category")

In [0]:
%sql
SELECT * FROM baraa_projectwork.silver.erp_product_category LIMIT 10